# RGO-driven run and HMI validation

Drive the model from the bundled RGO active-region record and compare the
modelled polar-field evolution against the observed HMI north-cap field over
cycle 24. This is the end-to-end BMR test: it exercises the Joy/Hale source
(`make_bmr` / `ARSource`), the poleward meridional flow, and the polar-cap
diagnostics together.

**What decides whether the pole reverses (and to which sign):**
* the meridional flow must be **poleward** (`peak_speed > 0`) to carry
  trailing-polarity flux to the poles — a negative value is equatorward and the
  pole never reverses;
* the **Hale polarity** must match the real cycle (odd cycles: N leading `+`;
  even cycles: N leading `-`) or the pole reverses to the *wrong* sign;
* `flux_scale` sets the amplitude (RGO |USFLUX| is ~10–50× too weak in absolute
  terms).

In [ ]:
import sys, pathlib
try:
    import sft2d
except ModuleNotFoundError:
    sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))

import numpy as np
import matplotlib.pyplot as plt

from sft2d import (create_grid, initialize_field, evolve, meridional_flow,
                   differential_rotation, calculate_polar_field, calculate_dm)
from sft2d.data import RGO_CSV, load_hmi_polar_field
from sft2d.src.ar_driver import ARSource

## Set up a cycle-24 driven run

The RGO record is turned into daily flux-normalised BMRs by `ARSource`. We seed
the observed pre-cycle-24 polarity (the HMI north cap is *negative* in 2010, so
the seed dipole is negative in the north).

In [ ]:
grid = create_grid(91, 180)
mf = meridional_flow(grid, peak_speed=15.0)      # +15 = poleward
dr = differential_rotation(grid)

flux_scale = 15.0
src = ARSource(str(RGO_CSV), start_date='2010-05-01', end_date='2023-01-01',
               flux_scale=flux_scale)
print(src.summary())

field = initialize_field(grid, 'dipole') * (-2.0)   # observed 2010 sign (N negative)

## Integrate and record the north-cap field

In [ ]:
y0 = 2010
yr, pnorth, psouth, dip, bfly = [], [], [], [], []
class Rec:
    def record(self, day, B):
        if day % 10 == 0:
            yr.append(y0 + 0.33 + day/365.25)
            n, s = calculate_polar_field(B, grid, pol_cap_extent_deg=20)
            pnorth.append(n); psouth.append(s); dip.append(calculate_dm(B, grid))
            bfly.append(B.mean(axis=1).copy())     # longitude-averaged B_r

Bf = evolve(field, grid, mf, dr, 2.5e8, src.num_days, source=src, recorder=Rec())
yr = np.array(yr); pnorth = np.array(pnorth); psouth = np.array(psouth)
model_bfly = np.array(bfly).T                       # (n_lat, n_time)
print(f"model N polar field: {pnorth[0]:+.2f} -> {pnorth[-1]:+.2f} G "
      f"({'REVERSED' if np.sign(pnorth[0])!=np.sign(pnorth[-1]) else 'no reversal'})")

## Overlay on the observed HMI north-cap field

In [ ]:
hmi = load_hmi_polar_field()          # dict of pandas Series + time (see docstring)
idx = hmi['mean_north'].index
hmi_yr = idx.year + (idx.dayofyear - 1)/365.25
mn, sn = hmi['mean_north'].values, hmi['std_north'].values
ms, ss = hmi['mean_south'].values, hmi['std_south'].values

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.fill_between(hmi_yr, mn-sn, mn+sn, color='C0', alpha=0.25)
ax.fill_between(hmi_yr, ms-ss, ms+ss, color='C2', alpha=0.25)
ax.plot(hmi_yr, mn, 'C0', lw=1, label='HMI north cap')
ax.plot(hmi_yr, ms, 'C2', lw=1, label='HMI south cap')
ax.plot(yr, pnorth, 'C0--', lw=2.5, label='model N')
ax.plot(yr, psouth, 'C2--', lw=2.5, label='model S')
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('year'); ax.set_ylabel('polar field [G]')
ax.set_title('Cycle 24 polar-field reversal: RGO-driven SFT vs HMI')
ax.legend(ncol=2, fontsize=8); fig.tight_layout()

## Notes for calibration

* Change `flux_scale` to scan the amplitude; `eta` (diffusivity) and `v0`
  (meridional-flow peak) shape the timing and the equator-to-pole transport.
* Flip `peak_speed` to `-15` to confirm the pole then fails to reverse — flow
  direction, not diffusivity, is what makes reversal possible at all.
* The same machinery scores against the polar-cap flux (`calculate_polar_flux`)
  for a quantitative misfit; that is the objective a parameter scan minimises.

## Butterfly comparison: model vs HMI

The HMI butterfly pickle holds `(bfly, time_years, sin_lat)` — its latitude axis
is uniform in **sine of latitude**, so plot it as `rad2deg(arcsin(sin_lat))`.
The model butterfly is uniform in latitude. Both are longitude-averaged $B_r$.

In [ ]:
from sft2d.data import load_hmi_butterfly
hmi_bfly, hmi_time, hmi_sinlat = load_hmi_butterfly()
model_lat = np.rad2deg(np.pi/2 - grid['colatitude'])

fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
vmax = 10
pm0 = ax[0].pcolormesh(yr, model_lat, model_bfly, cmap='RdBu_r',
                       vmin=-vmax, vmax=vmax, shading='auto')
ax[0].set_ylabel('latitude [deg]'); ax[0].set_ylim(-90, 90)
ax[0].set_title('model  <B_r>  (RGO-driven)'); fig.colorbar(pm0, ax=ax[0], label='G')

pm1 = ax[1].pcolormesh(hmi_time, np.rad2deg(np.arcsin(hmi_sinlat)), hmi_bfly,
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
ax[1].set_ylabel('latitude [deg]'); ax[1].set_xlabel('year'); ax[1].set_ylim(-90, 90)
ax[1].set_title('HMI  <B_r>'); fig.colorbar(pm1, ax=ax[1], label='G')
ax[1].set_xlim(2010.4, 2023.5)
fig.tight_layout()